# План ноутбука

## Цель  
Пошагово отладить пред-обработку одного КТ-исследования (DICOM-серия и NIfTI-файл).

## Шаги  

1. **Загрузка данных**  
   – загрузка 3D объёма из DICOM-папки и из `*.nii.gz`  
   – вывод формы, dtype, affine (для NIfTI)

2. **Приведение к HU + нормализация**  
   – окно HU [–1,000;400] → масштабирование в [0 ; 1]

3. **Унификация voxel-spacing**  
   – ресэмплинг к isotropic spacing 1.5 mm

4. **Кроп лёгких**  
   – быстрая mask-based bbox (Threshold HU<–300 → morphology)

5. **Шейпинг**  
   – `Resize/Pad` до фиксированного куба 128×128×128

6. **Tensor + сохранение**  
   – сохранить результат `SaveImage(writer="TorchTensor")`

7. **Визуализация чек-пойнтов**  
   – срезы, гистограммы интенсивности

## Методика  
-  один `monai.transforms.Compose`; после каждой трансформы — стоп-ячейка → смотрим вывод  
-  отладка на DICOM и NIfTI параллельно; затем масштабируем на весь датасет


# Шаг 0. Установка и импорт библиотек

In [ ]:
%pip install monai[all] itk-core pydicom nibabel SimpleITK

Defaulting to user installation because normal site-packages is not writeable
  Preparing metadata (pyproject.toml) ... done

[notice] A new release of pip is available: 23.0.1 -> 25.2
[notice] To update, run: python3 -m pip install --upgrade pip
ERROR: Exception:
Traceback (most recent call last):
  File "/kernel/lib/python3.10/site-packages/pip/_internal/cli/base_command.py", line 105, in _run_wrapper
    status = _inner_run()
  File "/kernel/lib/python3.10/site-packages/pip/_internal/cli/base_command.py", line 96, in _inner_run
    return self.run(options, args)
  File "/kernel/lib/python3.10/site-packages/pip/_internal/cli/req_command.py", line 68, in wrapper
    return func(self, options, args)
  File "/kernel/lib/python3.10/site-packages/pip/_internal/commands/install.py", line 387, in run
    requirement_set = resolver.resolve(
  File "/kernel/lib/python3.10/site-packages/pip/_internal/resolution/resolvelib/resolver.py", line 96, in resolve
    result = self._result = resolver.r

In [2]:
%pip uninstall -y itk itk-core itk-gdcm pydicom

Found existing installation: itk-core 5.4.4.post1
Uninstalling itk-core-5.4.4.post1:
  Successfully uninstalled itk-core-5.4.4.post1
Found existing installation: pydicom 3.0.1
Uninstalling pydicom-3.0.1:
  Successfully uninstalled pydicom-3.0.1


In [3]:
%pip install -qU itk pydicom


[notice] A new release of pip is available: 23.0.1 -> 25.2
[notice] To update, run: python3 -m pip install --upgrade pip


Чтобы импортировать модуль, нужно добавить ссыслку на него в path, это можно сделать из командной строки или с помощью sys.path.append, куда нужно передать путь к папке, где находится модуль

In [ ]:
# if not os.getcwd().endswith("chest-ct-classification"):
#     # для перемещения в корень репозитория
#     %cd {os.path.join(os.getcwd(), 'project', 'chest-ct-classification')}

In [ ]:
import sys
import os
import pathlib
import pandas as pd
sys.path.append(os.path.join(os.getcwd(), 'project', 'chest-ct-classification', 'src', 'CTPreprocessor'))  

# импортируем
from ct_preprocessor import PreprocConfig, MedPreprocessor

Failed to load image Python extension: '/usr/local/lib/python3.10/dist-packages/torchvision/image.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
2025-09-28 10:04:47.068182: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-09-28 10:04:49.347664: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the

In [24]:
# установим рабочую директорию
work_dir = os.path.join(os.path.expanduser('~'), 'work', 'data')

%cd {work_dir}

/home/jupyter/work/data


In [26]:
directory_path =  pathlib.Path("dataset_subset") 

# Получаем список только файлов, используя .iterdir() и .is_file()
file_list = [entry for entry in directory_path.iterdir()]

for i in file_list: print(i)

dataset_subset/normal
dataset_subset/protocols
dataset_subset/pathology
dataset_subset/test_subset_small


# Тест модуля

In [ ]:
# ================================
# ИМПОРТ ФИНАЛЬНОГО МОДУЛЯ
# ================================
import sys
from pathlib import Path
import torch
import gc

# Импорт финального модуля
# (сохраните код выше как ct_preprocessor_final.py)
from ct_preprocessor import PreprocConfig, MedPreprocessor

print("✅ Финальный модуль импортирован")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")


✅ Финальный модуль импортирован
PyTorch: 2.8.0+cu128
CUDA: False


In [ ]:
# ================================
# ФИНАЛЬНАЯ КОНФИГУРАЦИЯ
# ================================

# УСТАНОВИТЕ СВОЙ ПУТЬ
YOUR_DATA_PATH = '/home/jupyter/work/data/dataset_subset/test_subset_small'  # ИЗМЕНИТЕ НА СВОЙ ПУТЬ
OUTPUT_PATH = '/home/jupyter/work/data/preproc'

# Финальная проверенная конфигурация
final_config = PreprocConfig(
    input_root=YOUR_DATA_PATH,
    preproc_root=OUTPUT_PATH,
    
    # Умеренные параметры для надежности
    target_pixdim=(1.0, 1.0, 1.5),    # Не слишком агрессивный ресэмплинг
    target_size=(96, 96, 64),         # Разумный размер
    hu_window=(-1000.0, 400.0),       # Стандартное легочное окно
    padding_value=-2048.0,            # Обработка padding
    
    # ЛИБЕРАЛЬНЫЕ QA настройки (основаны на опыте)
    enable_qa=True,
    min_depth=16,                     # Минимум 16 срезов
    max_clip_share=0.95,             # 95% клиппинга OK
    max_padding_share=0.90,          # 90% воздуха OK
    
    # Отключенные фильтры (для максимальной совместимости)
    enable_dicom_filters=False,
    reject_localizer=False,
    require_ct_modality=False,
    
    # Производительность
    n_procs=4,                       # Консервативно для стабильности
    overwrite=True,
    
    # Подробное логирование
    enable_logging=True,
    log_level="INFO",
    pipeline_version="v3.0_final",
)

print("🔧 Финальная конфигурация готова")
print(f"   📂 Вход: {final_config.input_root}")
print(f"   📁 Выход: {final_config.preproc_root}")
print(f"   📏 Размер: {final_config.target_size}")
print(f"   🔍 QA: {'включена' if final_config.enable_qa else 'отключена'}")

🔧 Финальная конфигурация готова
   📂 Вход: /home/jupyter/work/data/dataset_subset/test_subset_small
   📁 Выход: /home/jupyter/work/data/preproc
   📏 Размер: (96, 96, 64)
   🔍 QA: включена


In [ ]:
# ================================
# ФИНАЛЬНОЕ ТЕСТИРОВАНИЕ
# ================================

print("🚀 ФИНАЛЬНОЕ ТЕСТИРОВАНИЕ")
print("="*50)

# Создание процессора
final_processor = MedPreprocessor(final_config)

# Запуск полного препроцессинга
print("🔄 Запуск препроцессинга...")
final_results = final_processor.build()

# Детальный анализ результатов
successful = sum(1 for r in final_results if r.get("ok", False))
cached = sum(1 for r in final_results if r.get("cached", False))
failed = len(final_results) - successful

print(f"\n📊 ФИНАЛЬНЫЕ РЕЗУЛЬТАТЫ:")
print(f"   ✅ Успешно: {successful}")
print(f"   💾 Из кэша: {cached}")
print(f"   ❌ Ошибок: {failed}")
print(f"   📈 Успешность: {successful/len(final_results)*100 if final_results else 0:.1f}%")

# Анализ типов ошибок
if failed > 0:
    print(f"\n🔍 АНАЛИЗ ОШИБОК:")
    error_types = {}
    for result in final_results:
        if not result.get("ok", False):
            reason = result.get("reason", "unknown")
            error_types[reason] = error_types.get(reason, 0) + 1
    
    for reason, count in error_types.items():
        print(f"   {reason}: {count}")

# Тестирование DataLoader
if successful > 0:
    print(f"\n🎯 ТЕСТИРОВАНИЕ DATALOADER:")
    try:
        train_loader = final_processor.get_dataloader(
            batch_size=2,
            num_workers=4,
            shuffle=True
        )
        
        print(f"   ✅ DataLoader создан: {len(train_loader)} батчей")
        
        # Тест первого батча
        for batch in train_loader:
            image = batch["image"]
            print(f"   📊 Батч: {image.shape}")
            print(f"   🔢 Тип: {image.dtype}")
            print(f"   📏 Диапазон: [{image.min():.3f}, {image.max():.3f}]")
            
            # Проверки
            assert image.shape[0] <= 2, "Размер батча"
            assert len(image.shape) == 5, "Размерность"
            assert 0.0 <= image.min() and image.max() <= 1.0, "Диапазон значений"
            
            print(f"   ✅ Все проверки пройдены!")
            break
            
    except Exception as e:
        print(f"   ❌ Ошибка DataLoader: {e}")

print(f"\n🎉 ФИНАЛЬНОЕ ТЕСТИРОВАНИЕ ЗАВЕРШЕНО!")

if successful > 0:
    print(f"✅ СИСТЕМА ГОТОВА К ОБУЧЕНИЮ АВТОЭНКОДЕРА!")
    print(f"   📁 Данные: {final_processor.output_dir}")
    print(f"   📊 Образцов: {successful}")
    print(f"   📏 Размер: {final_config.target_size}")
else:
    print(f"❌ ТРЕБУЕТСЯ ДОПОЛНИТЕЛЬНАЯ НАСТРОЙКА")
    print(f"   Попробуйте:")
    print(f"   - enable_qa=False")
    print(f"   - Другие пути к данным")
    print(f"   - Проверить формат файлов")


🚀 ФИНАЛЬНОЕ ТЕСТИРОВАНИЕ


2025-09-27 18:36:02,179 - MedPreprocessor - INFO - 🚀 MedPreprocessor v3.0 инициализирован
2025-09-27 18:36:02,180 - MedPreprocessor - INFO -    📂 /home/jupyter/work/data/dataset_subset/test_subset_small
2025-09-27 18:36:02,181 - MedPreprocessor - INFO -    📁 /home/jupyter/work/data/preproc/6b8cb4640fc9
2025-09-27 18:36:02,182 - MedPreprocessor - INFO - 🔍 Обнаружение данных...
2025-09-27 18:36:02,184 - MedPreprocessor - INFO - 🔍 Сканирование: /home/jupyter/work/data/dataset_subset/test_subset_small
2025-09-27 18:36:02,185 - MedPreprocessor - INFO - 📁 Поиск NIfTI...


🔄 Запуск препроцессинга...


2025-09-27 18:36:02,911 - MedPreprocessor - INFO -   ✅ NIfTI: 10
2025-09-27 18:36:02,911 - MedPreprocessor - INFO - 📁 Поиск DICOM...
2025-09-27 18:36:19,975 - MedPreprocessor - INFO -   ✅ DICOM: 6
2025-09-27 18:36:19,976 - MedPreprocessor - INFO - 🎯 Всего: 16
2025-09-27 18:36:19,978 - MedPreprocessor - INFO - 📊 К обработке: 16
Обработка:   0%|          | 0/16 [00:00<?, ?it/s]

  🔍 Вход: torch.Size([1, 1, 512, 512, 40]), мета: 43  🔍 Вход: torch.Size([1, 1, 512, 512, 39]), мета: 43

  🔍 Вход: torch.Size([1, 1, 512, 512, 35]), мета: 43
  🔍 Вход: torch.Size([1, 1, 512, 512, 43]), мета: 43
  🔄 Заменено padding: 2250080 вокселей
  🔄 Заменено padding: 1968820 вокселей  🔄 Заменено padding: 2193828 вокселей

  ⚠️ Нет данных spacing  🔄 Заменено padding: 2418836 вокселей
  ⚠️ Нет данных spacing

  ⚠️ Нет данных spacing
  ⚠️ Нет данных spacing
  📐 Resize: torch.Size([512, 512, 35]) -> (96, 96, 64)  📐 Resize: torch.Size([512, 512, 40]) -> (96, 96, 64)

  📐 Resize: torch.Size([512, 512, 39]) -> (96, 96, 64)
  📐 Resize: torch.Size([512, 512, 43]) -> (96, 96, 64)


...

  ✅ Ресэмплинг выполнен: torch.Size([1, 1, 351, 351, 203])
  ✅ Ресэмплинг выполнен: torch.Size([1, 1, 326, 326, 220])
  📐 Resize: torch.Size([351, 351, 203]) -> (96, 96, 64)
  📐 Resize: torch.Size([326, 326, 220]) -> (96, 96, 64)


Обработка:  88%|████████▊ | 14/16 [00:18<00:03,  1.89s/it]

  ✅ Ресэмплинг выполнен: torch.Size([1, 1, 400, 400, 240])
  📐 Resize: torch.Size([400, 400, 240]) -> (96, 96, 64)


Обработка: 100%|██████████| 16/16 [00:21<00:00,  1.32s/it]
2025-09-27 18:36:41,180 - MedPreprocessor - INFO - ✅ Завершено:
2025-09-27 18:36:41,181 - MedPreprocessor - INFO -    📊 Успешно: 16/16
2025-09-27 18:36:41,181 - MedPreprocessor - INFO -    💾 Кэш: 0
2025-09-27 18:36:41,195 - MedPreprocessor - INFO - 📚 DataLoader: 16 файлов



📊 ФИНАЛЬНЫЕ РЕЗУЛЬТАТЫ:
   ✅ Успешно: 16
   💾 Из кэша: 0
   ❌ Ошибок: 0
   📈 Успешность: 100.0%

🎯 ТЕСТИРОВАНИЕ DATALOADER:
   ✅ DataLoader создан: 8 батчей
   📊 Батч: torch.Size([2, 1, 96, 96, 64])
   🔢 Тип: torch.float32
   📏 Диапазон: [0.000, 1.000]
   ✅ Все проверки пройдены!

🎉 ФИНАЛЬНОЕ ТЕСТИРОВАНИЕ ЗАВЕРШЕНО!
✅ СИСТЕМА ГОТОВА К ОБУЧЕНИЮ АВТОЭНКОДЕРА!
   📁 Данные: /home/jupyter/work/data/preproc/6b8cb4640fc9
   📊 Образцов: 16
   📏 Размер: (96, 96, 64)


In [80]:
# Проверим все ли на месте
directory_path = Path("preproc/6b8cb4640fc9")

# Получаем список только файлов, используя .iterdir() и .is_file()
file_list = [entry for entry in directory_path.iterdir()]

# Выводим имена файлов
for file_path in file_list:
    print(file_path.name)

c8124ccd.nii.gz
1afb539f.nii.gz
28e195bf.nii.gz
f73d261b.nii.gz
c8124ccd.json
1afb539f.json
28e195bf.json
f73d261b.json
ff46b603.nii.gz
07cd2009.nii.gz
aa14c0fb.nii.gz
c3842cd2.nii.gz
ff46b603.json
07cd2009.json
aa14c0fb.json
c3842cd2.json
2af91d07.nii.gz
ed389786.nii.gz
2af91d07.json
ed389786.json
82761269.nii.gz
ad950838.nii.gz
ad950838.json
82761269.json
b1a19119.nii.gz
b1a19119.json
f645afb0.nii.gz
f645afb0.json
ec7b770d.nii.gz
ec7b770d.json
d7924d7c.nii.gz
d7924d7c.json
_index.json


# Обработка всех исследований и распределение по папкам

In [ ]:
# ================================
# МАССОВАЯ ОБРАБОТКА ПО КАТЕГОРИЯМ
# ================================

# Пути к данным
DATASET_ROOT = Path("/path/to/your/dataset_subset")  # ИЗМЕНИ
PREPROC_ROOT = Path("./preprocessed_data") # ИЗМЕНИ

def setup_preprocessing_configs():
    """Настройка конфигураций для каждой категории"""
    
    base_config = {
        "target_pixdim": (1.0, 1.0, 1.5),
        "target_size": (96, 96, 64),
        "hu_window": (-1000.0, 400.0),
        "padding_value": -2048.0,
        
        # Либеральные настройки для массовой обработки
        "enable_qa": True,
        "min_depth": 16,
        "max_clip_share": 0.95,
        "max_padding_share": 0.90,
        
        # Отключаем строгие фильтры
        "enable_dicom_filters": False,
        
        # Производительность
        "n_procs": 4,
        "overwrite": False,  # Не перезаписываем уже обработанные
        
        "enable_logging": True,
        "log_level": "INFO",
        "pipeline_version": "production_v1.0",
    }
    
    # Конфигурация для нормальных КТ
    normal_config = PreprocConfig(
        input_root=str(DATASET_ROOT / "normal"),
        preproc_root=str(PREPROC_ROOT / "normal"),
        **base_config
    )
    
    # Конфигурация для патологических КТ  
    pathology_config = PreprocConfig(
        input_root=str(DATASET_ROOT / "pathology"), 
        preproc_root=str(PREPROC_ROOT / "pathology"),
        **base_config
    )
    
    return normal_config, pathology_config

print("✅ Конфигурации для массовой обработки готовы")


In [ ]:
# ================================
# ОБРАБОТКА НОРМАЛЬНЫХ КТ (для обучения)
# ================================

print("🚀 НАЧИНАЕМ ОБРАБОТКУ НОРМАЛЬНЫХ КТ")
print("="*60)

normal_config, pathology_config = setup_preprocessing_configs()

# Обработка нормальных КТ
print("📋 Обрабатываем НОРМАЛЬНЫЕ КТ (для обучения автоэнкодера)")
normal_processor = MedPreprocessor(normal_config)
normal_results = normal_processor.build()

# Статистика
normal_success = sum(1 for r in normal_results if r.get("ok", False))
print(f"\n📊 РЕЗУЛЬТАТЫ ОБРАБОТКИ НОРМАЛЬНЫХ КТ:")
print(f"   ✅ Успешно: {normal_success}")
print(f"   ❌ Ошибок: {len(normal_results) - normal_success}")
print(f"   📈 Успешность: {normal_success/len(normal_results)*100:.1f}%")

if normal_success >= 350:
    print(f"   🎉 ОТЛИЧНО! {normal_success} нормальных КТ готовы для обучения")
    print(f"   📁 Сохранены в: {normal_processor.output_dir}")
else:
    print(f"   ⚠️ Мало данных для обучения: {normal_success} КТ")
    print(f"   Рекомендую: понизить пороги QA или проверить данные")


In [ ]:
# ================================
# ОБРАБОТКА ПАТОЛОГИЧЕСКИХ КТ (для тестирования)
# ================================

print("\n🚀 НАЧИНАЕМ ОБРАБОТКУ ПАТОЛОГИЧЕСКИХ КТ")
print("="*60)

# Обработка патологических КТ
print("🔬 Обрабатываем ПАТОЛОГИЧЕСКИЕ КТ (для тестирования anomaly detection)")
pathology_processor = MedPreprocessor(pathology_config)
pathology_results = pathology_processor.build()

# Статистика
pathology_success = sum(1 for r in pathology_results if r.get("ok", False))
print(f"\n📊 РЕЗУЛЬТАТЫ ОБРАБОТКИ ПАТОЛОГИЧЕСКИХ КТ:")
print(f"   ✅ Успешно: {pathology_success}")
print(f"   ❌ Ошибок: {len(pathology_results) - pathology_success}")
print(f"   📈 Успешность: {pathology_success/len(pathology_results)*100:.1f}%")

if pathology_success >= 350:
    print(f"   🎉 ОТЛИЧНО! {pathology_success} патологических КТ готовы для тестирования")
    print(f"   📁 Сохранены в: {pathology_processor.output_dir}")
else:
    print(f"   ⚠️ Мало данных для тестирования: {pathology_success} КТ")

In [ ]:
# ================================
# СОЗДАНИЕ DATALOADERS для обучения
# ================================

print("\n🎯 СОЗДАНИЕ DATALOADERS ДЛЯ ОБУЧЕНИЯ")
print("="*60)

if normal_success >= 10:
    # DataLoader только для нормальных КТ (для обучения автоэнкодера)
    train_loader = normal_processor.get_dataloader(
        batch_size=4,
        num_workers=2,
        shuffle=True
    )
    
    # Валидационный DataLoader (также только нормальные)
    val_size = max(2, normal_success // 5)  # 20% для валидации
    val_loader = normal_processor.get_dataloader(
        batch_size=2,
        num_workers=1,
        shuffle=False
    )
    
    print(f"🎯 DATALOADERS ГОТОВЫ:")
    print(f"   📚 Train loader: {len(train_loader)} батчей по 4 КТ")
    print(f"   🧪 Val loader: {len(val_loader)} батчей по 2 КТ")
    
    # Тестируем первый батч
    print(f"\n🔍 ТЕСТИРОВАНИЕ ПЕРВОГО БАТЧА:")
    for batch in train_loader:
        images = batch["image"]
        print(f"   📊 Размер батча: {images.shape}")
        print(f"   🔢 Тип данных: {images.dtype}")
        print(f"   📏 Диапазон: [{images.min():.3f}, {images.max():.3f}]")
        
        # Проверки корректности
        assert images.shape[1:] == (1, 96, 96, 64), "Неправильный размер КТ"
        assert 0.0 <= images.min() and images.max() <= 1.0, "Неправильный диапазон"
        
        print(f"   ✅ Все проверки пройдены!")
        break
        
    print(f"\n🎉 СИСТЕМА ПОЛНОСТЬЮ ГОТОВА К ОБУЧЕНИЮ АВТОЭНКОДЕРА!")
    
else:
    print(f"❌ Недостаточно нормальных КТ для обучения")
    print(f"   Нужно минимум 10, найдено: {normal_success}")


In [ ]:
# ================================
# ЯЧЕЙКА 5: СОЗДАНИЕ ТЕСТОВЫХ DATALOADERS (норма + патология)
# ================================

print("\n🧪 СОЗДАНИЕ ТЕСТОВЫХ DATALOADERS")
print("="*60)

# Функция для создания смешанного тестового набора
def create_test_dataloader(normal_processor, pathology_processor, batch_size=2):
    """Создает DataLoader с нормальными и патологическими КТ для тестирования"""
    
    # Загружаем индексы обработанных файлов
    normal_index_file = normal_processor.output_dir / "_index.json"
    pathology_index_file = pathology_processor.output_dir / "_index.json"
    
    test_files = []
    test_labels = []  # 0 = норма, 1 = патология
    
    # Нормальные файлы
    if normal_index_file.exists():
        with open(normal_index_file, 'r') as f:
            normal_index = json.load(f)
        
        normal_files = [
            item["output_img"] for item in normal_index["items"]
            if item.get("ok", False) and "output_img" in item
        ]
        
        test_files.extend(normal_files)
        test_labels.extend([0] * len(normal_files))
        print(f"   📁 Нормальных КТ для теста: {len(normal_files)}")
    
    # Патологические файлы  
    if pathology_index_file.exists():
        with open(pathology_index_file, 'r') as f:
            pathology_index = json.load(f)
            
        pathology_files = [
            item["output_img"] for item in pathology_index["items"] 
            if item.get("ok", False) and "output_img" in item
        ]
        
        test_files.extend(pathology_files)
        test_labels.extend([1] * len(pathology_files))
        print(f"   🔬 Патологических КТ для теста: {len(pathology_files)}")
    
    print(f"   🎯 Всего для тестирования: {len(test_files)} КТ")
    
    return test_files, test_labels

# Создаем тестовый набор
if pathology_success > 0:
    test_files, test_labels = create_test_dataloader(
        normal_processor, pathology_processor
    )
    
    print(f"\n✅ ТЕСТОВЫЙ НАБОР ГОТОВ:")
    print(f"   📊 Норма: {test_labels.count(0)} КТ")
    print(f"   🔬 Патология: {test_labels.count(1)} КТ")
    print(f"   ⚖️ Баланс: {test_labels.count(1)/len(test_labels)*100:.1f}% патологии")
    
    # Сохраняем для использования в модели
    test_info = {
        "files": test_files,
        "labels": test_labels,
        "normal_count": test_labels.count(0),
        "pathology_count": test_labels.count(1)
    }
    
    import json
    with open(PREPROC_ROOT / "test_dataset_info.json", 'w') as f:
        json.dump(test_info, f, indent=2)
        
    print(f"   💾 Информация о тесте сохранена: test_dataset_info.json")

print(f"\n🏁 МАССОВАЯ ОБРАБОТКА ЗАВЕРШЕНА!")
print(f"📈 ИТОГОВАЯ СТАТИСТИКА:")
print(f"   🟢 Нормальных КТ: {normal_success} (для обучения)")
print(f"   🔴 Патологических КТ: {pathology_success} (для тестирования)")
print(f"   📁 Данные готовы в: {PREPROC_ROOT}")